# Companion Notebook — Chapter 10: AI Foundations Capstone
## Smart Sensor Data Quality Monitor
**Artificial Intelligence Learning Hub** — *Level 1 AI Foundations Capstone*

Notebook pendamping ini memandu Anda mengeksekusi *Reference Implementation* resmi Chapter 10:
1. **Pembangkitan Data Telemetri Sintetis Deterministik** (Bab 07 & 08)
2. **Validasi Skema & Batas Fisika Domain** (Bab 03 & 08)
3. **Analisis Statistik Deskriptif & Pagar Tukey IQR** (Bab 06)
4. **Mesin Inferensi First-Order Logic & Peringatan Mutu** (Bab 05)
5. **Kalkulasi Data Quality Index (DQI) & Visualisasi Dashboard** (Bab 07 & 09)
6. **Refleksi Batasan Sistem & Pengawasan Manusia (*Human-in-the-Loop*)**


## 1. Pembangkitan Data Telemetri Sintetis
Kita membangkitkan 120 sampel telemetri multi-sensor sintetis (`TEMP-01`, `HUMID-01`, `VIB-01`) dengan *random seed* tetap (`seed=42`).

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Pastikan modul kode bab 10 dapat diimpor
sys.path.insert(0, str(Path("../code").resolve()))

from dataset_generator import generate_synthetic_sensor_data

raw_csv = Path("../code/data/sensor_telemetry_raw.csv")
df_raw = generate_synthetic_sensor_data(raw_csv, n_records=120, seed=42)

print(f"Total Sampel Telemetri: {len(df_raw)} baris")
print(f"Sensor Teridentifikasi: {df_raw['sensor_id'].unique().tolist()}")
print(f"Missing Values (NaN) Terkontrol: {df_raw['measurement_value'].isna().sum()} sel")
print("\n5 Baris Pertama:")
print(df_raw.head())


Total Sampel Telemetri: 120 baris
Sensor Teridentifikasi: ['TEMP-01', 'HUMID-01', 'VIB-01']
Missing Values (NaN) Terkontrol: 3 sel

5 Baris Pertama:
             timestamp sensor_id  measurement_value  unit
0  2026-11-05 08:00:00   TEMP-01              28.75  degC
1  2026-11-05 08:00:00  HUMID-01              64.45   %RH
2  2026-11-05 08:00:00    VIB-01               2.46  mm/s
3  2026-11-05 08:05:00   TEMP-01              30.29  degC
4  2026-11-05 08:05:00  HUMID-01              64.06   %RH


## 2. Validasi Skema & Batas Fisika Pengukuran
Sensor fisik dapat mengalami kegagalan transmisi (nilai kosong) atau korsleting listrik (nilai di luar batas fisika, misal suhu di bawah nol mutlak). Kita memvalidasi setiap rekaman tanpa membuang datanya untuk menjaga jejak audit.

In [2]:
from validator import validate_schema, validate_sensor_records

is_valid, msg = validate_schema(df_raw)
df_val, report = validate_sensor_records(df_raw)

print(f"Status Skema: {is_valid} ({msg})")
print(f"Rekaman Berstatus VALID : {report['valid_records_count']} baris")
print(f"Rekaman MISSING         : {report['missing_count']} baris")
print(f"Rekaman INVALID_BOUNDS  : {report['invalid_bounds_count']} baris")
print("\nDetail Pelanggaran Batas Fisik:")
for inv in report["invalid_details"]:
    print(f"- [{inv['timestamp']}] {inv['sensor_id']}: {inv['value']} -> {inv['reason']}")


Status Skema: True (Skema valid)
Rekaman Berstatus VALID : 115 baris
Rekaman MISSING         : 3 baris
Rekaman INVALID_BOUNDS  : 2 baris

Detail Pelanggaran Batas Fisik:
- [2026-11-05 08:40:00] TEMP-01: -300.0 -> Nilai -300.0 di luar batas fisika yang dimungkinkan [-10.0, 100.0]
- [2026-11-05 09:50:00] TEMP-01: 145.0 -> Nilai 145.0 di luar batas fisika yang dimungkinkan [-10.0, 100.0]


## 3. Analisis Statistik Deskriptif & Pagar Tukey (IQR)
Hanya dari data yang berstatus `VALID`, kita menghitung statistik deskriptif dan pagar Tukey ($Q_1 - 1.5\text{IQR}$ dan $Q_3 + 1.5\text{IQR}$) untuk memisahkan anomali statistik dari fluktuasi normal.

In [3]:
from statistics_analyzer import compute_sensor_statistics

stats = compute_sensor_statistics(df_val)

print("RINGKASAN STATISTIK DESKRIPTIF PER SENSOR:")
print("-" * 70)
print(f"{'Sensor':<11} {'Satuan':<8} {'Mean':<7} {'Median':<7} {'Std':<7} {'IQR':<7} {'Pagar Tukey':<18} {'Outliers'}")
for sid, s in sorted(stats.items()):
    fences = f"[{s['lower_fence']}, {s['upper_fence']}]"
    print(f"{sid:<11} {s['unit']:<8} {s['mean']:<7} {s['median']:<7} {s['std']:<7} {s['iqr']:<7} {fences:<18} {s['outlier_count']}")
print("-" * 70)


RINGKASAN STATISTIK DESKRIPTIF PER SENSOR:
----------------------------------------------------------------------
Sensor      Satuan   Mean    Median  Std     IQR     Pagar Tukey        Outliers
HUMID-01    %RH      64.92   65.24   4.14    5.36    [54.75, 76.21]     0
TEMP-01     degC     28.68   28.05   6.01    1.29    [25.31, 30.49]     2
VIB-01      mm/s     2.14    2.12    0.43    0.58    [0.96, 3.29]       0
----------------------------------------------------------------------


## 4. Mesin Inferensi Rule-Based & Data Quality Index (DQI)
Menggunakan aturan deduktif First-Order Logic, sistem mengklasifikasikan rekaman menjadi alert peringatan kausal transparan dan menghitung skor DQI per instrumen sensor.

In [4]:
from rule_engine import DataQualityRuleEngine

engine = DataQualityRuleEngine(stats)
alerts = engine.evaluate_records(df_val)
dqi_scores = engine.calculate_data_quality_index(df_val)

print(f"Total Alert Terpicu: {len(alerts)} peringatan\n")
print("SKOR DATA QUALITY INDEX (DQI):")
for sid, sc in dqi_scores.items():
    status = "EXCELLENT" if sc >= 85 else "ACCEPTABLE" if sc >= 70 else "POOR"
    print(f"- {sid}: {sc:.2f} / 100 [{status}]")

print("\nSampel 3 Peringatan Pertama:")
for a in alerts[:3]:
    print(f"* [{a['severity']}] {a['rule_id']}: {a['title']}")
    print(f"  Penjelasan : {a['explanation']}")
    print(f"  Rekomendasi: {a['recommended_action']}")


Total Alert Terpicu: 7 peringatan

SKOR DATA QUALITY INDEX (DQI):
- HUMID-01: 97.50 / 100 [EXCELLENT]
- TEMP-01: 85.00 / 100 [EXCELLENT]
- VIB-01: 100.00 / 100 [EXCELLENT]

Sampel 3 Peringatan Pertama:
* [WARNING] RULE-01-MISSING-BURST: Paket Data Telemetri Hilang
  Penjelasan : Nilai sensor TEMP-01 kosong pada timestamp 2026-11-05 08:20:00. Kemungkinan gangguan jaringan LoRa/Wi-Fi.
  Rekomendasi: Periksa konektivitas jaringan gateway dan buffer memori sensor.
* [CRITICAL] RULE-02-PHYSICAL-IMPOSSIBLE: Kegagalan Fisik / Kerusakan Sensor Terdeteksi
  Penjelasan : Nilai -300.0 berada di luar batas operasional fisika sistem untuk TEMP-01.
  Rekomendasi: Kirim teknisi lapangan untuk kalibrasi perangkat keras atau periksa kabel grounding sensor.
* [WARNING] RULE-01-MISSING-BURST: Paket Data Telemetri Hilang
  Penjelasan : Nilai sensor TEMP-01 kosong pada timestamp 2026-11-05 09:15:00. Kemungkinan gangguan jaringan LoRa/Wi-Fi.
  Rekomendasi: Periksa konektivitas jaringan gateway dan buffer me

## 5. Visualisasi Dashboard Mutu Sensor
Kita merender citra dashboard kualitas multi-panel menggunakan Matplotlib dalam mode *headless* (`Agg` backend) dan menyimpannya ke `sensor_quality_dashboard.png`.

In [5]:
from visualization import render_quality_dashboard

dashboard_out = Path("../code/data/sensor_quality_dashboard.png")
render_quality_dashboard(df_val, stats, dqi_scores, dashboard_out)
print(f"[SUCCESS] Dashboard berhasil diekspor ke: {dashboard_out.resolve()}")


[SUCCESS] Dashboard berhasil diekspor ke: D:\Workstation\Repo My Corses\Kecerdasan Buatan\content\level-1-ai-foundations\chapter-10\code\data\sensor_quality_dashboard.png


## 6. Ringkasan & Refleksi Etis

Melalui praktikum Capstone ini, Anda telah membuktikan bagaimana pilar-pilar Level 1 bekerja secara harmonis:
- **Logika Deterministik**: Memberikan batas keselamatan yang mutlak dan terjelaskan.
- **Statistik Deskriptif**: Memandu pemahaman distribusi normal tanpa overengineering.
- **Pengawasan Manusia**: Peringatan sistem bersifat *advisory tool* (alat bantu), di mana keputusan akhir operasional tetap berada di tangan insinyur manusia.

Selamat! Anda telah siap melangkah ke **Level 2: Machine Learning (Chapter 11)**.